In [ ]:
import cv2
import numpy as np

# ================= CONFIG =================
IP_WEBCAM_URL = "http://10.56.95.217:8080/video"  # CHANGE THIS
FRAME_W, FRAME_H = 640, 480

# ================= FINGER DETECTION =================
def is_finger_present(frame, red_ratio_thresh=1.2, min_red=80):
    r = np.mean(frame[:, :, 2])
    g = np.mean(frame[:, :, 1])
    b = np.mean(frame[:, :, 0])
    red_ratio = r / (g + b + 1e-6)
    return (r > min_red) and (red_ratio > red_ratio_thresh)

# ================= ILLUMINATION CHECK (UN-COVERED ONLY) =================
def check_illumination_uncovered(frame, min_brightness=80):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    brightness = np.mean(gray)
    if brightness < min_brightness:
        return False, "Turn ON the light"
    return True, "Illumination OK"

# ================= FINGER PLACEMENT =================
def check_finger_placement(frame,
                           edge_thresh=1.1,
                           corner_thresh=1.25,
                           var_thresh=18):
    h, w, _ = frame.shape

    # ---------- Edge regions ----------
    edge_regions = {
        "move left": frame[:, :w//4],
        "move right": frame[:, 3*w//4:],
        "move up": frame[:h//4, :],
        "move down": frame[3*h//4:, :]
    }

    issues = []

    for direction, region in edge_regions.items():
        r = np.mean(region[:, :, 2])
        g = np.mean(region[:, :, 1])
        b = np.mean(region[:, :, 0])
        red_ratio = r / (g + b + 1e-6)

        if red_ratio < edge_thresh:
            issues.append(direction)

    # ---------- Corner regions (STRICT) ----------
    cs = min(h, w) // 10  # corner size (~10%)
    corners = [
        frame[:cs, :cs],             # top-left
        frame[:cs, -cs:],            # top-right
        frame[-cs:, :cs],            # bottom-left
        frame[-cs:, -cs:]             # bottom-right
    ]

    for corner in corners:
        r = np.mean(corner[:, :, 2])
        g = np.mean(corner[:, :, 1])
        b = np.mean(corner[:, :, 0])
        red_ratio = r / (g + b + 1e-6)

        gray = cv2.cvtColor(corner, cv2.COLOR_BGR2GRAY)
        variance = np.var(gray)

        # Either low redness OR high variance → gap
        if red_ratio < corner_thresh or variance > var_thresh:
            return False, "Cover all corners completely"

    if not issues:
        return True, "Finger placement OK"

    return False, "Finger not covering properly → " + ", ".join(issues)


# ================= MAIN LOOP =================
cap = cv2.VideoCapture(IP_WEBCAM_URL)

if not cap.isOpened():
    print("❌ Cannot connect to IPWebCam")
    raise SystemExit

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (FRAME_W, FRAME_H))

    finger_present = is_finger_present(frame)

    # ---- Illumination Logic ----
    if finger_present:
        illum_msg = "Finger detected (PPG mode)"
        illum_color = (0, 255, 0)
    else:
        illum_ok, illum_msg = check_illumination_uncovered(frame)
        illum_color = (0, 255, 0) if illum_ok else (0, 0, 255)

    # ---- Placement Logic ----
    place_ok, place_msg = check_finger_placement(frame)
    place_color = (0, 255, 0) if place_ok else (0, 0, 255)

    # ---- Display ----
    cv2.putText(frame, illum_msg, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, illum_color, 2)
    cv2.putText(frame, place_msg, (10, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, place_color, 2)

    cv2.imshow("Camera PPG Finger Guidance", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
